# 🏛️ PROYEK AKHIR UTS KECERDASAN BUATAN (KELAS A)
## Sistem Perencanaan Otomatis Rute & Jadwal Wisata Kota Batu Berbasis Algoritma STRIPS

**Kelompok 8:**
1. **Mutiara Putri Faradilla** (NIM: 240605110113)
2. **Vanya Nyssa Maulida** (NIM: 240605110104)

**Dosen Pengampu:** Hani Nurhayati, M.T.  
**Program Studi:** Teknik Informatika, UIN Maulana Malik Ibrahim Malang

---
### 📖 Deskripsi Proyek
Notebook ini mengimplementasikan algoritma **STRIPS (Stanford Research Institute Problem Solver)** berbasis logika predikat (*First-Order Logic*) untuk menyelesaikan permasalahan perencanaan perjalanan wisata multi-kendala secara otomatis (*Tour Guide Itinerary Planning*) di Kota Batu, Jawa Timur.

In [ ]:
# ==============================================================================
# 1. BASIS PENGETAHUAN & ENGINE STRIPS MURNI (ZERO-DEPENDENCY)
# ==============================================================================
import heapq
from dataclasses import dataclass
from typing import List, Dict, Set, Optional, Tuple

@dataclass
class Destination:
    id: str
    name: str
    open_time: int      # menit dari 00:00 (misal 08:30 = 510)
    close_time: int     # menit dari 00:00 (misal 16:30 = 990)
    ticket_price: int   # Rupiah
    duration: int       # Durasi kunjungan dalam menit
    category: str
    is_attraction: bool = True

DESTINATIONS: Dict[str, Destination] = {
    "Stasiun_Malang": Destination("Stasiun_Malang", "Stasiun Kota Malang (Kotabaru)", 0, 1440, 0, 0, "transport", is_attraction=False),
    "Hotel_Batu": Destination("Hotel_Batu", "Pusat Hotel & Penginapan Kota Batu", 0, 1440, 0, 0, "penginapan", is_attraction=False),
    "Jatim_Park_2": Destination("Jatim_Park_2", "Jatim Park 2 (Batu Secret Zoo)", 510, 990, 110000, 180, "edukasi"),
    "Museum_Angkut": Destination("Museum_Angkut", "Museum Angkut & Pasar Apung", 720, 1200, 100000, 150, "edukasi"),
    "Jatim_Park_3": Destination("Jatim_Park_3", "Jatim Park 3 (Dino Park & Millennial Glow)", 660, 1260, 100000, 150, "edukasi"),
    "Paralayang_Batu": Destination("Paralayang_Batu", "Wisata Paralayang & Omah Kayu", 420, 1080, 25000, 90, "alam"),
    "Coban_Rondo": Destination("Coban_Rondo", "Air Terjun Coban Rondo & Labirin", 450, 1020, 35000, 120, "alam"),
    "San_Terra": Destination("San_Terra", "Flora Wisata San Terra de Laponte", 480, 1020, 30000, 120, "alam"),
    "Selecta": Destination("Selecta", "Taman Rekreasi Selecta", 420, 1020, 50000, 120, "alam"),
    "Petik_Apel_Kusuma": Destination("Petik_Apel_Kusuma", "Agrowisata Petik Apel Kusuma", 480, 1020, 60000, 90, "alam"),
    "Coban_Rais": Destination("Coban_Rais", "Coban Rais & Batu Flower Garden", 480, 960, 35000, 120, "alam"),
    "BNS": Destination("BNS", "Batu Night Spectacular (BNS)", 900, 1380, 40000, 120, "hiburan"),
    "Alun_Alun_Batu": Destination("Alun_Alun_Batu", "Alun-Alun Kota Batu & Pos Ketan", 0, 1440, 0, 60, "hiburan"),
    "Brawijaya_Oleh_Oleh": Destination("Brawijaya_Oleh_Oleh", "Pusat Oleh-Oleh Istana Brawijaya", 480, 1320, 0, 45, "belanja")
}

TRAVEL_TIMES = {
    ("Stasiun_Malang", "Hotel_Batu"): 45, ("Stasiun_Malang", "Jatim_Park_2"): 50,
    ("Hotel_Batu", "Jatim_Park_2"): 15, ("Hotel_Batu", "Museum_Angkut"): 12,
    ("Hotel_Batu", "Alun_Alun_Batu"): 8, ("Hotel_Batu", "BNS"): 16,
    ("Hotel_Batu", "Selecta"): 25, ("Hotel_Batu", "Paralayang_Batu"): 30,
    ("Jatim_Park_2", "Museum_Angkut"): 12, ("Jatim_Park_2", "BNS"): 8,
    ("Museum_Angkut", "Alun_Alun_Batu"): 10, ("Alun_Alun_Batu", "BNS"): 15,
    ("BNS", "Brawijaya_Oleh_Oleh"): 10, ("Brawijaya_Oleh_Oleh", "Hotel_Batu"): 12
}

def get_travel_time(origin: str, dest: str) -> int:
    if origin == dest: return 0
    if (origin, dest) in TRAVEL_TIMES: return TRAVEL_TIMES[(origin, dest)]
    if (dest, origin) in TRAVEL_TIMES: return TRAVEL_TIMES[(dest, origin)]
    return 20

def fmt_time(minutes: int) -> str:
    h = (minutes // 60) % 24
    m = minutes % 60
    return f"{h:02d}:{m:02d} WIB"

@dataclass
class WorldState:
    current_location: str
    current_time: int
    current_budget: int
    visited: Set[str]
    tickets: Set[str]
    had_lunch: bool
    had_dinner: bool

    def clone(self) -> 'WorldState':
        return WorldState(
            self.current_location, self.current_time, self.current_budget,
            set(self.visited), set(self.tickets), self.had_lunch, self.had_dinner
        )

@dataclass
class STRIPSOperator:
    name: str
    action_type: str
    target: str
    preconditions_desc: str
    add_list_desc: str
    delete_list_desc: str
    duration_cost: int
    money_cost: int

    def is_applicable(self, state: WorldState, max_end_time: int = 1320) -> Tuple[bool, str]:
        if state.current_budget < self.money_cost:
            return False, "Budget tidak cukup"
        if self.action_type == "TRAVEL":
            arrival = state.current_time + get_travel_time(state.current_location, self.target)
            if arrival > max_end_time: return False, "Melebihi batas waktu"
            target_dest = DESTINATIONS[self.target]
            if target_dest.is_attraction and (arrival + 30 < target_dest.open_time):
                return False, f"Datang terlalu cepat (buka {fmt_time(target_dest.open_time)})"
            return True, "OK"
        elif self.action_type == "BUY_TICKET":
            return (state.current_location == self.target and self.target not in state.tickets), "OK"
        elif self.action_type == "VISIT":
            dest = DESTINATIONS[self.target]
            if state.current_location != self.target: return False, "Beda lokasi"
            if dest.ticket_price > 0 and self.target not in state.tickets: return False, "Belum beli tiket"
            if state.current_time < dest.open_time: return False, "Belum buka"
            if state.current_time + dest.duration > dest.close_time: return False, "Tutup sebelum selesai"
            if self.target in state.visited: return False, "Sudah dikunjungi"
            return True, "OK"
        elif self.action_type == "EAT":
            return True, "OK"
        return False, "Error"

    def apply(self, state: WorldState) -> WorldState:
        nxt = state.clone()
        nxt.current_budget -= self.money_cost
        nxt.current_time += self.duration_cost
        if self.action_type == "TRAVEL": nxt.current_location = self.target
        elif self.action_type == "BUY_TICKET": nxt.tickets.add(self.target)
        elif self.action_type == "VISIT":
            nxt.visited.add(self.target)
            if self.target in nxt.tickets: nxt.tickets.remove(self.target)
        elif self.action_type == "EAT":
            if self.target == "LUNCH": nxt.had_lunch = True
            elif self.target == "DINNER": nxt.had_dinner = True
        return nxt

class STRIPSTourPlanner:
    def __init__(self, start_loc: str, start_time: int, budget: int, goals: List[str], final_loc: str = "Hotel_Batu", max_time: int = 1320):
        self.initial_state = WorldState(start_loc, start_time, budget, set(), set(), False, False)
        self.goals = set(goals)
        self.final_location = final_loc
        self.max_end_time = max_time

    def is_goal_satisfied(self, s: WorldState) -> bool:
        return self.goals.issubset(s.visited) and s.current_location == self.final_location

    def generate_operators(self, s: WorldState) -> List[STRIPSOperator]:
        ops = []
        unvisited = self.goals - s.visited
        if s.current_location in unvisited:
            d = DESTINATIONS[s.current_location]
            if d.ticket_price > 0 and (s.current_location not in s.tickets):
                ops.append(STRIPSOperator(f"BUY_TICKET({d.id})", "BUY_TICKET", d.id, f"AT({d.id}) ∧ ¬HAS_TICKET ∧ BUDGET≥{d.ticket_price}", f"HAS_TICKET({d.id})", "BUDGET", 5, d.ticket_price))
            if (d.ticket_price == 0) or (s.current_location in s.tickets):
                ops.append(STRIPSOperator(f"VISIT({d.id})", "VISIT", d.id, f"AT({d.id}) ∧ HAS_TICKET", f"VISITED({d.id})", f"HAS_TICKET({d.id})", d.duration, 0))
        for g in unvisited:
            if g != s.current_location:
                ops.append(STRIPSOperator(f"TRAVEL({s.current_location}, {g})", "TRAVEL", g, f"AT({s.current_location})", f"AT({g})", f"AT({s.current_location})", get_travel_time(s.current_location, g), 0))
        if len(unvisited) == 0 and s.current_location != self.final_location:
            ops.append(STRIPSOperator(f"TRAVEL({s.current_location}, {self.final_location})", "TRAVEL", self.final_location, "ALL_VISITED", f"AT({self.final_location})", f"AT({s.current_location})", get_travel_time(s.current_location, self.final_location), 0))
        if not s.had_lunch and (690 <= s.current_time <= 840):
            ops.append(STRIPSOperator("EAT(LUNCH)", "EAT", "LUNCH", "¬HAD_LUNCH ∧ TIME(11:30-14:00)", "HAD_LUNCH", "BUDGET", 45, 35000))
        return ops

    def solve(self):
        pq = [(0, 0, 0, self.initial_state, [])]
        visited_history = set()
        counter = 0
        while pq:
            f, cost, _, curr_s, steps = heapq.heappop(pq)
            if self.is_goal_satisfied(curr_s): return steps
            s_key = (curr_s.current_location, tuple(sorted(curr_s.visited)), tuple(sorted(curr_s.tickets)), curr_s.had_lunch, curr_s.current_time // 15)
            if s_key in visited_history: continue
            visited_history.add(s_key)
            for op in self.generate_operators(curr_s):
                ok, _ = op.is_applicable(curr_s, self.max_end_time)
                if not ok: continue
                nxt_s = op.apply(curr_s)
                step = {
                    "no": len(steps) + 1,
                    "action": op.name,
                    "location": DESTINATIONS[nxt_s.current_location].name,
                    "start": fmt_time(curr_s.current_time),
                    "end": fmt_time(nxt_s.current_time),
                    "cost": op.money_cost,
                    "budget": nxt_s.current_budget
                }
                h = len(self.goals - nxt_s.visited) * 60 + (20 if nxt_s.current_location != self.final_location else 0)
                counter += 1
                heapq.heappush(pq, (nxt_s.current_time + h, nxt_s.current_time, counter, nxt_s, steps + [step]))
        return None

print("✅ Modul STRIPS Tour Planner Berhasil Dimuat!")

In [ ]:
#@title 🧭 Form Uji Coba Parameter Rencana Perjalanan
import pandas as pd

Titik_Kedatangan = "Stasiun Kota Malang (Kotabaru)" #@param ["Stasiun Kota Malang (Kotabaru)", "Pusat Hotel & Penginapan Kota Batu"]
Jam_Mulai = "08:30" #@param ["07:30", "08:00", "08:30", "09:00"]
Uang_Saku = 500000 #@param {type:"integer"}
Lokasi_Akhir = "Pusat Hotel & Penginapan Kota Batu" #@param ["Pusat Hotel & Penginapan Kota Batu", "Stasiun Kota Malang (Kotabaru)"]

Kunjungi_Jatim_Park_2 = True #@param {type:"boolean"}
Kunjungi_Museum_Angkut = True #@param {type:"boolean"}
Kunjungi_BNS = True #@param {type:"boolean"}
Kunjungi_Selecta = False #@param {type:"boolean"}
Kunjungi_Paralayang = False #@param {type:"boolean"}

start_id = "Stasiun_Malang" if "Stasiun" in Titik_Kedatangan else "Hotel_Batu"
final_id = "Hotel_Batu" if "Hotel" in Lokasi_Akhir else "Stasiun_Malang"
h, m = map(int, Jam_Mulai.split(":"))
start_min = h * 60 + m

goals = []
if Kunjungi_Jatim_Park_2: goals.append("Jatim_Park_2")
if Kunjungi_Museum_Angkut: goals.append("Museum_Angkut")
if Kunjungi_BNS: goals.append("BNS")
if Kunjungi_Selecta: goals.append("Selecta")
if Kunjungi_Paralayang: goals.append("Paralayang_Batu")

print(f"⚙️ Menjalankan STRIPS Solver untuk {len(goals)} target destinasi...")
planner = STRIPSTourPlanner(start_id, start_min, Uang_Saku, goals, final_id)
plan = planner.solve()

if plan:
    print("\n🎉 RENCANA PERJALANAN OPTIMAL DITEMUKAN:\n")
    df = pd.DataFrame(plan)
    df.columns = ["Langkah", "Aksi STRIPS", "Lokasi", "Mulai", "Selesai", "Biaya (Rp)", "Sisa Saldo (Rp)"]
    display(df)
else:
    print("❌ Tidak ditemukan solusi yang memenuhi batasan waktu / budget.")